# Making OCR for L25

Chay EasyOCR (GPU) tren ``L25_data/keyframes``, xuat file JSON ra ``L25_data/ocr``.

- **Reader**: EasyOCR (ngon ngu `vi` + `en`)
- **Device**: CUDA neu co, fallback CPU
- **Output**: `{video_id}.json` trong thu muc `ocr`
- **Skip**: bo qua video da co file JSON

In [1]:
import gc
import csv
import json
import re
from pathlib import Path

import cv2
import torch
from easyocr import Reader
from tqdm.auto import tqdm

BASE_DIR = Path.cwd().parent
KEYFRAME_DIR = BASE_DIR / "L25_data" / "keyframes"
OCR_DIR = BASE_DIR / "L25_data" / "ocr"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
OCR_LANGUAGES = ["vi", "en"]
MIN_CONFIDENCE = 0.30
SKIP_EXISTING_OUTPUT = True
NUM_READER_WORKERS = 2

d:\Truong_pipeline\My_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not KEYFRAME_DIR.exists():
    raise FileNotFoundError(f"Khong tim thay thu muc keyframe: {KEYFRAME_DIR}")


def natural_sort_key(path: Path):
    # Sort video/frame dirs numerically, e.g. V2 before V10
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", path.name)]


video_dirs = sorted(
    (p for p in KEYFRAME_DIR.iterdir() if p.is_dir()), key=natural_sort_key
)
video_ids = [d.name for d in video_dirs]

OCR_DIR.mkdir(parents=True, exist_ok=True)
print(f"Tong so video: {len(video_dirs)}")
print(f"Output OCR: {OCR_DIR}")
print(f"Videos (first 5): {video_ids[:5]}")

Tong so video: 88
Output OCR: d:\Truong_pipeline\My_Code\L25_data\ocr
Videos (first 5): ['L25_V001', 'L25_V002', 'L25_V003', 'L25_V004', 'L25_V005']


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
reader = Reader(OCR_LANGUAGES, gpu=(device == "cuda"))

print(f"Using device: {device}")
print(f"Reader workers: {NUM_READER_WORKERS}")

Using device: cuda
Reader workers: 2


In [4]:
def ocr_one_image(image_path):
    # Run EasyOCR on one image, return list of text records
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError("Khong doc duoc anh")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    detections = reader.readtext(
        image_rgb,
        detail=1,
        paragraph=False,
        workers=NUM_READER_WORKERS,
    )

    records = []
    for box, text, conf in detections:
        conf = float(conf)
        if conf < MIN_CONFIDENCE:
            continue
        records.append({
            "text": text.strip(),
            "confidence": round(conf, 4),
            "bbox": [[float(x), float(y)] for x, y in box],
        })

    return records

In [5]:
def export_video_ocr(video_id):
    # OCR tung video, xuat file CSV (chi frame_id + text)
    video_dir = KEYFRAME_DIR / video_id
    if not video_dir.is_dir():
        raise FileNotFoundError(f"Khong tim thay thu muc video: {video_dir}")

    output_file = OCR_DIR / f"{video_id}.csv"
    if SKIP_EXISTING_OUTPUT and output_file.exists() and output_file.stat().st_size > 0:
        print(f"[SKIP] {video_id}: da co file CSV")
        return output_file

    image_files = sorted(
        (p for p in video_dir.iterdir()
         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
        key=natural_sort_key,
    )
    if not image_files:
        raise FileNotFoundError(f"Khong tim thay anh nao trong {video_dir}")

    rows = []
    pbar = tqdm(image_files, desc=f"OCR {video_id}", leave=False)
    for image_path in pbar:
        try:
            frame_no = int(image_path.stem)
        except ValueError:
            continue

        records = ocr_one_image(image_path)
        for rec in records:
            rows.append([frame_no, rec["text"]])

    if not rows:
        raise ValueError(f"Khong co text nao duoc OCR cho {video_id}")

    rows.sort(key=lambda r: r[0])

    import csv as _csv
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = _csv.writer(f)
        writer.writerow(["n", "text"])
        writer.writerows(rows)

    print(f"[DONE] {video_id} -> {output_file.relative_to(BASE_DIR)} (rows: {len(rows)})")
    return output_file


# Chay het tat ca video trong thu muc keyframes cua L25
processed_count = 0
skipped_count = 0
failed_count = 0

video_iter = tqdm(video_ids, desc="OCR L25 videos")
for video_id in video_iter:
    output_file = OCR_DIR / f"{video_id}.csv"

    if SKIP_EXISTING_OUTPUT and output_file.exists() and output_file.stat().st_size > 0:
        skipped_count += 1
        video_iter.set_postfix(done=processed_count, skip=skipped_count, fail=failed_count)
        continue

    try:
        export_video_ocr(video_id)
        processed_count += 1
    except Exception as e:
        failed_count += 1
        print(f"[{video_id}] FAIL: {e}")

    video_iter.set_postfix(done=processed_count, skip=skipped_count, fail=failed_count)

    # Giai phong GPU memory giua cac video
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

print(f"\nHoan tat. Tong video: {len(video_ids)} | DONE: {processed_count} | SKIP: {skipped_count} | FAIL: {failed_count}")
print(f"Output OCR nam trong: {OCR_DIR}")

# Neu muon chay rieng tung video:
# export_video_ocr("L25_V001")
# export_video_ocr("L25_V002")

OCR L25 videos:   0%|          | 0/88 [4:23:04<?, ?it/s]


KeyboardInterrupt: 

## Tom tac

- **Input**: keyframe trong `L25_data/keyframes/{video_id}/`
- **Output**: file CSV tai `L25_data/ocr/{video_id}.csv` (mot file cho moi video)
- **CSV format**: 2 cot
  - `n`: frame_id (so thu tu frame trong video)
  - `text`: noi dung text OCR (da duoc strip va filter confidence >= 0.30)
- **Toi uu hieu nang**:
  - Dung GPU (CUDA) qua EasyOCR
  - Song song da luong qua `workers` parameter
  - Progress bar `tqdm` (theo tung anh va theo tung video)
  - `torch.cuda.empty_cache()` + `gc.collect()` giua cac video de tranh memory leak
  - Natural sort dam bao thu tu frame dung